In [ ]:
import os
from typing import Any, cast

import matplotlib.pyplot as plt
import torch
import torchvision.transforms.v2 as T
from diffusers import StableDiffusionInstructPix2PixPipeline
from diffusers.utils import logging
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.io import read_image
from torchvision.utils import save_image
from tqdm.auto import tqdm

from attn_ctrl import register_attention_controller
from pipeline_iter import IterEditPipeline
from pipeline_iterdiff import AttentionStore, IterDiffPipeline

logging.set_verbosity_error()
torch.enable_grad(False)

DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
MODEL_ID = "timbrooks/instruct-pix2pix"

In [ ]:
class IterEditDataset(Dataset):
    def __init__(self, image_dir: str, transform=None):
        self.instructions = [
            {
                # teaser,
                "image_path": "28000/28234.png",
                "instructions": [
                    "Change the gender to male.",
                    "Change the hair color to gray.",
                    "Put on a scarf.",
                    "Make the person's skin tanned.",
                    "Add a beard to the face.",
                ],
            },
            {
                # sample 1
                "image_path": "20000/20728.png",
                "instructions": [
                    "Make the face look older.",
                    "Change the gender to female.",
                    "Put on sunglasses.",
                    "Add mustache to the face.",
                    "Change the hair color to black.",
                ],
            },
            {
                # sample 2
                "image_path": "55000/55496.png",
                "instructions": [
                    "Give an Asian appearance.",
                    "Make the person's skin tanned.",
                    "Change the gender to male.",
                    "Change the hair color to black.",
                    "Put on glasses.",
                ],
            },
            {
                # sample 3
                "image_path": "17000/17168.png",
                "instructions": [
                    "Put on a cap.",
                    "Change the gender to male.",
                    "Add a mustache to the face.",
                    "Make the hair curlier.",
                    "Make the person's skin darker.",
                ],
            },
        ]

        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.instructions)

    def __getitem__(self, idx):
        image_path, insts = self.instructions[idx].values()

        image = read_image(os.path.join(self.image_dir, image_path))
        if self.transform:
            image = self.transform(image)
        return image, insts


dataset = Subset(
    IterEditDataset(
        image_dir=os.path.join("datasets", "ffhq", "images1024x1024"),
        transform=T.Compose(
            [
                T.Resize(512, antialias=True),
                T.ToDtype(torch.float32, scale=True),
            ]
        ),
    ),
    indices=[
        0,
    ],
)


def collate_fn(batch):
    images, insts = zip(*batch)
    images = torch.stack(images)
    insts = list(list(i) for i in zip(*insts))
    return images, insts


dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
)

In [ ]:
def show_result(
    images: torch.Tensor,
    results: torch.Tensor,
    insts: list[str],
):
    for orig_image, edited_images, inst in zip(
        images.unbind(dim=0), results.unbind(dim=0), zip(*insts)
    ):
        fig, ax = plt.subplots(1, edited_images.shape[0] + 1, figsize=(20, 10))
        ax[0].imshow(orig_image.permute(1, 2, 0).cpu().numpy())
        ax[0].axis("off")
        ax[0].set_title("Original")
        for i, (img, ins) in enumerate(zip(edited_images, inst), start=1):
            ax[i].imshow(img.permute(1, 2, 0).cpu().numpy())
            ax[i].axis("off")
            ax[i].set_title(ins)
        fig.tight_layout()
        fig.show()


In [ ]:
os.makedirs("sample_image", exist_ok=True)

vanilla_pipe = IterEditPipeline(
    cast(
        StableDiffusionInstructPix2PixPipeline,
        StableDiffusionInstructPix2PixPipeline.from_pretrained(
            MODEL_ID, torch_dtype=torch.float32, safety_checker=None
        ),
    )
)
vanilla_pipe = vanilla_pipe.to(DEVICE)

for image, insts in tqdm(dataloader):
    image = image.to(DEVICE)
    insts = insts

    results = vanilla_pipe(
        prompt=insts,
        image=image,
        num_inference_steps=100,
        guidance_scale=7.5,
        image_guidance_scale=1.5,
        generator=torch.Generator().manual_seed(123),
    )

    show_result(image, results, insts)


vanilla_pipe = vanilla_pipe.to("cpu")

save_image(image[0], os.path.join("sample_image", "ip2p_0.png"))
for i, img in enumerate(results[0]):
    save_image(img, os.path.join("sample_image", f"ip2p_{i + 1}.png"))

iterdiff_pipe = IterEditPipeline(
    cast(
        IterDiffPipeline,
        IterDiffPipeline.from_pretrained(
            MODEL_ID, torch_dtype=torch.float32, safety_checker=None
        ),
    )
)
iterdiff_pipe = iterdiff_pipe.to(DEVICE)

for j, (image, insts) in enumerate(tqdm(dataloader)):
    kwargs_list: list[dict[str, Any]] = [
        {"mb_size": 40, "mb_save_topk": 20},
    ]

    for kw in tqdm(kwargs_list, leave=False):
        controller = AttentionStore(
            size=iterdiff_pipe.pipe.unet.sample_size,
            **kw,
        )
        register_attention_controller(iterdiff_pipe.pipe.unet, controller)
        controller.full_reset()

        image = image.to(DEVICE)
        insts = insts

        results = iterdiff_pipe(
            prompt=insts,
            image=image,
            num_inference_steps=100,
            guidance_scale=7.5,
            image_guidance_scale=1.5,
            generator=torch.Generator().manual_seed(123),
            attn_ctrl=controller,
            use_scfg=True,
            use_factor=True,
        )

        show_result(image, results, insts)

iterdiff_pipe = iterdiff_pipe.to("cpu")

save_image(image[0], os.path.join("sample_image", "iter_0.png"))
for i, img in enumerate(results[0]):
    save_image(img, os.path.join("sample_image", f"iter_{i + 1}.png"))
